# Colab 32 — SHAP cho baseline LR đã khóa

Giải thích **outer-fold models** của quy trình Colab 31 (`lr_ba_inner_selected`).
Giữ nguyên grid C/weight/feature selection, seed 42, threshold 0.5, không SMOTENC.
Không dùng SHAP để chọn biến hoặc sửa baseline. Hungarian không tải/đánh giá.

## Thiết kế giải thích
- Mỗi outer hospital: chọn cấu hình trong hai source hospitals, fit source, giải thích outer test.
- SHAP LinearExplainer với Independent masker; background là **toàn bộ real source train**
  sau preprocessing. Không subsample ngầm, không lấy background từ test.
- Output là log-odds của lớp dương: base_value + tổng SHAP = decision_function;
  sigmoid của tổng này bằng xác suất lớp dương. SHAP không phải phần trăm xác suất.
- Giả định interventional/independent không mô hình hóa tương quan giữa biến.
  Với one-hot, phân rã theo cột rồi cộng SHAP có dấu về biến gốc; không phải conditional/group SHAP.
- Missing indicators giữ thành nhóm riêng `missing::<feature>`.
- Importance theo site là mean(abs(grouped SHAP)); macro importance lấy trung bình đều các site.
  Nếu một biến không được chọn ở fold, đóng góp của model đó bằng 0 và được đánh dấu selected=False.
- Mỗi site chọn tối đa 2 FP có xác suất cao nhất và 2 FN có xác suất thấp nhất;
  tie theo row_id. Đây là minh họa lỗi cực đoan theo quy tắc cố định, không đại diện mọi ca lỗi.
- Giải thích đóng góp vào dự đoán, không suy luận nguyên nhân gây bệnh hoặc giá trị lâm sàng.

Nguồn: [SHAP LinearExplainer documentation](https://shap.readthedocs.io/en/stable/generated/shap.LinearExplainer.html).

In [ ]:
!pip -q install "scikit-learn==1.6.1" "imbalanced-learn==0.13.0" "optuna==4.2.1" seaborn
!pip -q install "shap==0.48.0"

: 

In [ ]:
import json
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import seaborn as sns
from IPython.display import display
from imblearn.over_sampling import SMOTENC
from scipy.special import expit
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("default")
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.max_columns", 100)

FEATURES = [
    "age", "sex", "cp", "trestbps", "chol", "fbs", "restecg",
    "thalach", "exang", "oldpeak", "slope", "ca", "thal",
]
TARGET = "target"
SITE = "site"
NUMERICAL_FEATURES = ["age", "trestbps", "chol", "thalach", "oldpeak"]
CATEGORICAL_FEATURES = [
    "sex", "cp", "fbs", "restecg", "exang", "slope", "ca", "thal",
]

BASE_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease"
FILES = {
    "cleveland": "processed.cleveland.data",
    "switzerland": "processed.switzerland.data",
    "va": "processed.va.data",
}
COLUMNS = FEATURES + ["num"]
DEVELOPMENT_SITES = ["cleveland", "switzerland", "va"]

MODEL_SEEDS = (42,)
N_TRIALS = 10
THRESHOLD = 0.50

OUTPUT_DIR = Path("/content/uci_multicenter_shap_lab32_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_DATA_DIR_CANDIDATES = [
    Path("/content/heart-disease-diagnosis/data/raw/uci_multicenter"),
    Path("/content/data/raw/uci_multicenter"),
    Path("data/raw/uci_multicenter"),
    Path("../data/raw/uci_multicenter"),
]
LOCAL_DATA_DIR = next(
    (path for path in LOCAL_DATA_DIR_CANDIDATES if path.exists()),
    None,
)

print("Development sites:", DEVELOPMENT_SITES)
print("Model seeds:", MODEL_SEEDS)
print("External holdout loaded:", False)
from sklearn.metrics import balanced_accuracy_score
import platform, importlib.metadata, shutil

C_GRID = (0.01, 0.1, 1.0, 10.0)
# Numeric weights are {0: 1, 1: weight}; balanced is learned from each train fold.
WEIGHT_GRID = (None, 'balanced', 0.25, 0.5, 0.75, 1.5, 2.0, 4.0)
THRESHOLD = 0.5
LOCAL_DATA_DIR_CANDIDATES += [Path('../../data/raw/uci_multicenter')]
LOCAL_DATA_DIR = next((p for p in LOCAL_DATA_DIR_CANDIDATES if p.exists()), None)
SEED = 42
warnings.filterwarnings("ignore", category=DeprecationWarning)
import shap
from scipy.special import expit

In [ ]:
def read_uci(site, filename):
    source = (LOCAL_DATA_DIR / filename) if LOCAL_DATA_DIR else f"{BASE_URL}/{filename}"
    frame = pd.read_csv(
        source,
        names=COLUMNS,
        na_values=["?"],
        skipinitialspace=True,
    )
    frame = frame.apply(pd.to_numeric, errors="coerce")
    assert frame["num"].notna().all()
    assert frame["num"].isin([0, 1, 2, 3, 4]).all()
    frame[TARGET] = (frame["num"] > 0).astype("int8")
    frame[SITE] = site
    return frame[FEATURES + [TARGET, SITE]]


development = pd.concat(
    [read_uci(site, filename) for site, filename in FILES.items()],
    ignore_index=True,
)

assert len(development) == 626
assert set(development[SITE]) == set(DEVELOPMENT_SITES)

site_summary = development.groupby(SITE).agg(
    rows=(TARGET, "size"),
    positives=(TARGET, "sum"),
    positive_rate=(TARGET, "mean"),
).reset_index()

missing_by_site = development.groupby(SITE)[FEATURES].apply(
    lambda frame: frame.isna().mean()
).T

print("Data source:", str(LOCAL_DATA_DIR) if LOCAL_DATA_DIR else "UCI URL fallback")
print("Development shape:", development.shape)
display(site_summary.round(4))
display(pd.crosstab(development[SITE], development[TARGET], margins=True))
display((missing_by_site * 100).round(1))

site_summary.to_csv(OUTPUT_DIR / "development_site_summary.csv", index=False)
missing_by_site.to_csv(OUTPUT_DIR / "development_missing_by_site.csv")

In [ ]:
def apply_p1(frame):
    out = frame.copy()
    for column in FEATURES:
        out[column] = pd.to_numeric(out[column], errors="coerce")

    # P1: sentinel đã được đọc thành NaN; thêm rule có căn cứ cho giá trị không hợp lệ.
    for column in ["trestbps", "chol"]:
        out.loc[out[column] <= 0, column] = np.nan

    return out

In [ ]:
def fit_controlled(train, config, C, ratio, seed, return_details=False):
    features = [f for f in FEATURES if not (config['drop'] and f in ['ca', 'thal'])]
    numeric = [f for f in NUMERICAL_FEATURES if f in features]
    categorical = [f for f in CATEGORICAL_FEATURES if f in features]
    raw = apply_p1(train)[numeric + categorical]
    fill = {}
    for f in raw:
        values = raw[f].dropna()
        fill[f] = (values.median() if f in numeric else values.mode().iloc[0]) if len(values) else 0.0
    scaler = StandardScaler().fit(raw[numeric].fillna(fill))
    def transform(frame):
        x = apply_p1(frame)[numeric + categorical]
        pieces = [scaler.transform(x[numeric].fillna(fill)), x[categorical].fillna(fill).to_numpy()]
        if config['indicators']:
            pieces.append(x.isna().astype(float).to_numpy())
        return np.column_stack(pieces)
    x = transform(train)
    y = train[TARGET].to_numpy()
    cat_indices = list(range(len(numeric), x.shape[1]))
    if config['sampling']:
        counts = np.bincount(y, minlength=2)
        # Do not undersample when requested ratio is already reached.
        if counts.min() < 2:
            raise ValueError('SMOTENC requires at least two samples per class')
        if counts.min() / counts.max() < ratio:
            x, y = SMOTENC(categorical_features=cat_indices, sampling_strategy=ratio,
                k_neighbors=min(5, int(counts.min()) - 1), random_state=seed).fit_resample(x, y)
    encoder = ColumnTransformer([
        ('numeric', 'passthrough', list(range(len(numeric)))),
        ('categorical', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_indices),
    ])
    model = LogisticRegression(C=C, class_weight=config.get('class_weight', 'balanced' if config['balanced'] else None),
                               solver='lbfgs', max_iter=3000, random_state=seed)
    model.fit(encoder.fit_transform(x), y)
    predict = lambda frame: model.predict_proba(encoder.transform(transform(frame)))[:, 1]
    if return_details:
        input_groups = categorical + (['missing::'+f for f in numeric+categorical] if config['indicators'] else [])
        groups = list(numeric)
        for group, categories in zip(input_groups, encoder.named_transformers_['categorical'].categories_):
            groups.extend([group]*len(categories))
        names = encoder.get_feature_names_out().tolist()
        assert len(groups)==len(names)==len(model.coef_[0])
        return dict(model=model, transform=lambda frame: encoder.transform(transform(frame)),
                    predict=predict, groups=groups, names=names)
    return predict

def metrics(y, p, threshold=0.5):
    pred = (np.asarray(p) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(accuracy=accuracy_score(y, pred), balanced_accuracy=balanced_accuracy_score(y, pred),
        recall=recall_score(y, pred, zero_division=0), specificity=tn/(tn+fp),
        precision=precision_score(y, pred, zero_division=0), f1=f1_score(y, pred, zero_division=0),
        roc_auc=roc_auc_score(y, p), average_precision=average_precision_score(y, p),
        brier=brier_score_loss(y, p), tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp), n=len(y))

In [ ]:
def config_for(drop=False, weight=None, sampling=False):
    return dict(drop=drop, sampling=sampling, balanced=False, indicators=True,
                class_weight=weight if weight in (None, 'balanced') else {0:1.0, 1:float(weight)})

def inner_scores(source, config, C, seed):
    rows=[]
    for hospital in sorted(source[SITE].unique()):
        train=source[source[SITE]!=hospital]
        valid=source[source[SITE]==hospital]
        assert set(train.index).isdisjoint(valid.index)
        assert train[SITE].nunique()==source[SITE].nunique()-1 and valid[SITE].nunique()==1
        predict=fit_controlled(train, config, C, 1.0, seed)
        rows.append(dict(inner_site=hospital, **metrics(valid[TARGET],predict(valid))))
    return rows

def select_candidate(candidates, objective):
    secondary='min_balanced_accuracy' if objective=='balanced_accuracy' else 'macro_accuracy'
    return max(candidates, key=lambda c:(c['macro_'+objective],c[secondary]))

def tune_branch(source, drop, weights, sampling, objective, seed):
    candidates=[]
    for C in C_GRID:
        for weight in weights:
            config=config_for(drop,weight,sampling)
            folds=inner_scores(source,config,C,seed)
            candidates.append(dict(C=C,weight=weight,drop=drop,sampling=sampling,
                macro_accuracy=float(np.mean([f['accuracy'] for f in folds])),
                macro_balanced_accuracy=float(np.mean([f['balanced_accuracy'] for f in folds])),
                min_balanced_accuracy=float(min(f['balanced_accuracy'] for f in folds)),
                folds=folds))
    return select_candidate(candidates,objective),candidates
def select_baseline(source, seed=SEED):
    candidates=[]
    for drop in (False,True):
        _, branch_candidates=tune_branch(source,drop,WEIGHT_GRID,False,'balanced_accuracy',seed)
        candidates.extend(branch_candidates)
    return select_candidate(candidates,'balanced_accuracy'),candidates

def flatten_candidates(candidates,best,stage):
    return [dict(stage=stage,selected=c is best,
        **{k:v for k,v in c.items() if k!='folds'},**fold)
        for c in candidates for fold in c['folds']]

## Nested LOCO + SHAP + vérification additivity

In [ ]:
ALL_GROUPS=FEATURES+['missing::'+f for f in FEATURES]
metric_rows,importance_rows,coefficient_rows,prediction_rows,selection_rows=[],[],[],[],[]
grouped_frames,case_rows,background_rows=[],[],[]
MAX_CASES_PER_ERROR=2
for hospital in DEVELOPMENT_SITES:
    train=development[development[SITE]!=hospital]
    test=development[development[SITE]==hospital]
    assert set(train.index).isdisjoint(test.index)
    best,_=select_baseline(train)
    selection_rows.append(dict(site=hospital,**{k:v for k,v in best.items() if k!='folds'}))
    details=fit_controlled(train,config_for(best['drop'],best['weight']),best['C'],1.0,SEED,return_details=True)
    background=details['transform'](train)
    x=details['transform'](test)
    masker=shap.maskers.Independent(background,max_samples=len(background))
    explainer=shap.LinearExplainer(details['model'],masker)
    explanation=explainer(x)
    values=np.asarray(explanation.values)
    base=np.broadcast_to(np.asarray(explanation.base_values), (len(test),))
    logits=details['model'].decision_function(x)
    probability=details['predict'](test)
    assert np.allclose(base+values.sum(axis=1),logits,atol=1e-8)
    assert np.allclose(expit(base+values.sum(axis=1)),probability,atol=1e-8)
    assert np.allclose(values,(x-background.mean(axis=0))*details['model'].coef_[0],atol=1e-8)
    group_names=list(dict.fromkeys(details['groups']))
    grouped=np.column_stack([values[:,np.array(details['groups'])==g].sum(axis=1) for g in group_names])
    assert np.allclose(grouped.sum(axis=1),values.sum(axis=1),atol=1e-8)
    grouped_exp=shap.Explanation(values=grouped,base_values=base,feature_names=group_names)
    site_dir=OUTPUT_DIR/hospital
    site_dir.mkdir(exist_ok=True)
    np.savez_compressed(site_dir/'encoded_shap.npz',values=values,base_values=base,
        transformed_test=x,feature_names=np.array(details['names']),groups=np.array(details['groups']),row_ids=test.index.to_numpy())
    for row_id in train.index:background_rows.append(dict(explained_site=hospital,background_row_id=int(row_id)))
    gf=pd.DataFrame(grouped,index=test.index,columns=group_names).reindex(columns=ALL_GROUPS,fill_value=0.0)
    gf.insert(0,'row_id',test.index)
    gf.insert(0,'site',hospital)
    grouped_frames.append(gf)
    for g in ALL_GROUPS:
        importance_rows.append(dict(site=hospital,feature=g,selected=g in group_names,
            mean_abs_shap=float(gf[g].abs().mean()),mean_signed_shap=float(gf[g].mean())))
    for name,group,coef,importance in zip(details['names'],details['groups'],details['model'].coef_[0],np.abs(values).mean(axis=0)):
        coefficient_rows.append(dict(site=hospital,encoded_feature=name,group=group,
            coefficient=float(coef),mean_abs_shap=float(importance)))
    metric_rows.append(dict(site=hospital,**metrics(test[TARGET],probability,THRESHOLD)))
    cases=pd.DataFrame(dict(row_id=test.index,target=test[TARGET].to_numpy(),probability=probability))
    cases['prediction']=(cases.probability>=THRESHOLD).astype(int)
    cases['outcome']=np.where(cases.target==cases.prediction,np.where(cases.target==1,'TP','TN'),np.where(cases.target==0,'FP','FN'))
    prediction_rows.extend(cases.assign(site=hospital).to_dict('records'))
    plt.figure()
    shap.plots.beeswarm(grouped_exp,max_display=20,show=False)
    plt.title(hospital+' — grouped SHAP (log-odds); no feature-value coloring')
    plt.tight_layout()
    plt.savefig(site_dir/'summary_beeswarm.png',dpi=160,bbox_inches='tight')
    plt.close('all')
    for error in ['FP','FN']:
        chosen=cases[cases.outcome==error].sort_values(['probability','row_id'],ascending=[error=='FN',True]).head(MAX_CASES_PER_ERROR)
        for _,case in chosen.iterrows():
            row_id=int(case.row_id)
            pos=test.index.get_loc(row_id)
            case_rows.append(dict(site=hospital,**case.to_dict(),base_log_odds=float(base[pos]),
                raw_features=json.dumps({f:None if pd.isna(test.loc[row_id,f]) else float(test.loc[row_id,f]) for f in FEATURES})))
            shap.plots.waterfall(grouped_exp[pos],max_display=15,show=False)
            plt.title(f'{hospital} {error} row={row_id} p={case.probability:.3f}')
            plt.savefig(site_dir/f'{error}_{row_id}_waterfall.png',dpi=160,bbox_inches='tight')
            plt.close('all')
    print('Completed SHAP + additivity:',hospital,flush=True)

In [ ]:
site_metrics=pd.DataFrame(metric_rows)
importance=pd.DataFrame(importance_rows)
coefficients=pd.DataFrame(coefficient_rows)
macro_importance=importance.groupby('feature').agg(mean_abs_shap=('mean_abs_shap','mean'),
    selected_sites=('selected','sum')).sort_values('mean_abs_shap',ascending=False).reset_index()
predictions=pd.DataFrame(prediction_rows)
assert len(predictions)==626 and predictions.row_id.nunique()==626
display(site_metrics.round(4))
display(macro_importance.round(4))
display(pd.DataFrame(case_rows))
top=macro_importance.head(20).feature
heat=importance.pivot(index='feature',columns='site',values='mean_abs_shap').loc[top]
plt.figure(figsize=(9,9))
sns.heatmap(heat,annot=True,fmt='.3f',cmap='Blues')
plt.title('Mean absolute grouped SHAP by hospital (log-odds)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR/'importance_by_hospital.png',dpi=160)
plt.show()
for name,frame in dict(site_metrics=site_metrics,macro_metrics=site_metrics.drop(columns='site').mean().to_frame('macro').reset_index(),
    site_importance=importance,macro_importance=macro_importance,encoded_coefficients=coefficients,
    outer_predictions=predictions,grouped_shap=pd.concat(grouped_frames,ignore_index=True),
    selected_configs=pd.DataFrame(selection_rows),error_cases=pd.DataFrame(case_rows),
    background_membership=pd.DataFrame(background_rows)).items():
    frame.to_csv(OUTPUT_DIR/(name+'.csv'),index=False)
protocol=dict(notebook='Lab32',baseline='Colab31 lr_ba_inner_selected',C_grid=C_GRID,weight_grid=WEIGHT_GRID,
    seed=SEED,threshold=THRESHOLD,background='all real source train, patient weighted',
    explainer='LinearExplainer / Independent masker',output_units='positive-class log-odds',
    grouping='sum signed encoded SHAP within raw variable; missing indicators separate',
    case_rule='up to 2 highest-probability FP and 2 lowest-probability FN per site; row_id tie',
    aggregation='equal site mean absolute grouped SHAP; absent groups zero with selected flag',
    limitations='model and background differ by fold; descriptive, not causal; correlated features not modeled',
    external_holdout_loaded=False,additivity_checks_passed=True,
    versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','shap']})
(OUTPUT_DIR/'shap_protocol.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
print('ZIP:',shutil.make_archive(str(OUTPUT_DIR),'zip',OUTPUT_DIR))

## Cách viết Results / Discussion
- Báo cáo top features riêng từng hospital trước khi tổng hợp macro.
- Đối chiếu `encoded_coefficients.csv`: coefficient thể hiện hướng tác động trên thang encoded;
  SHAP còn phụ thuộc độ lệch của giá trị so với background. Không cộng hệ số one-hot thành một hệ số biến gốc.
- Background/model thay đổi theo fold, nên chênh lệch SHAP giữa hospitals không chỉ do population shift.
- Beeswarm gộp không tô màu theo giá trị vì một nhóm có thể gồm nhiều encoded columns.
- `error_cases.csv` là minh họa hậu kiểm các lỗi đã chọn bằng quy tắc, không phải evidence causal.
- SHAP không cải thiện metric; nếu sửa model dựa vào kết quả giải thích thì đó là nghiên cứu mới.